<a href="https://colab.research.google.com/github/tensorbytes0202/Deep-learning/blob/main/FC_GAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter

In [20]:
class Discriminator(nn.Module):
  def __init__(self,in_features):
    super().__init__()
    self.disc = nn.Sequential(
        nn.Linear(image_dim,128),
        nn.LeakyReLU(0.1),
        nn.Linear(128,1),
        nn.Sigmoid(),

    )

  def forward(self,x):
    return self.disc(x)

class Generator(nn.Module):
  def __init__(self,z_dim,img_dim):
    super().__init__()
    self.gen = nn.Sequential(
        nn.Linear(z_dim,256),
        nn.LeakyReLU(0.1),
        nn.Linear(256,img_dim),
        nn.Tanh(),
  )

  def forward(self,x):
    return self.gen(x)

In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter

#Hyperparameter etc.

device ="cuda" if torch.cuda.is_available() else "cpu"
lr = 1e-4
z_dim = 64 #128,256
image_dim = 28*28*1
batch_size = 32
num_epochs = 50

disc = Discriminator(image_dim).to(device)
gen = Generator(z_dim,image_dim).to(device)
fixed_noise = torch.randn((batch_size,z_dim)).to(device)
transform_pipeline = transforms.Compose(
    [transforms.ToTensor(),transforms.Normalize((0.1307,),(0.3801,))]
)

In [26]:
dataset=datasets.MNIST(root="dataset/",transform=transform_pipeline,download=True)
loader = DataLoader(dataset,batch_size=batch_size,shuffle=True)
opt_disc = optim.Adam(disc.parameters(),lr=lr)
opt_gen = optim.Adam(gen.parameters(),lr=lr)
criterion = nn.BCELoss()
writer_fake = SummaryWriter(f"runs/GAN_MNIST/fake")
writer_real = SummaryWriter(f"runs/GAN_MNIST/real")
step = 0

In [ ]:
for epoch in range(num_epochs):
  for batch_idx, (real,_) in enumerate(loader):
    real = real.view(-1, image_dim).to(device) # Move real to device
    batch_size = real.shape[0]

    ### Train Discriminator: max log(D(real)) + log(1- D(G(z)))
    noise = torch.randn(batch_size,z_dim).to(device)
    fake = gen(noise) # Keep fake here without detach for now to build its graph for Generator training

    # Train with real images
    disc_real_output = disc(real).view(-1)
    lossD_real = criterion(disc_real_output,torch.ones_like(disc_real_output))

    # Train with fake images (detach fake from generator's graph)
    disc_fake_output = disc(fake.detach()).view(-1)
    lossD_fake = criterion(disc_fake_output,torch.zeros_like(disc_fake_output))

    lossD = (lossD_real + lossD_fake)/2
    disc.zero_grad()
    lossD.backward(retain_graph=True) # retain_graph=True because 'fake' is needed for generator's backward pass in the next cell
    opt_disc.step()

In [ ]:
### Train Discriminator: min log(1- D(G(z))) <--> max log(D(G(z)))
output = disc(fake).view(-1)
lossG = criterion(output,torch.ones_like(output))
gen.zero_grad()
lossG.backward()
opt_gen.step()

if batch_idx == 0:
  print(
      f"Epoch [{epoch}/{num_epochs}] Batch {batch_idx}/{len(loader)} \
      Loss D: {lossD:.4f}, loss G: {lossG:.4f}"
  )

In [ ]:
with torch.no_grad():
  fake = gen(fixed_noise).reshape(-1,1,28,28)
  data = real.reshape(-1,1,28,28)
  img_grid_fake = torchvision.utils.make_grid(fake,normalize=True)
  img_grid_real = torchvision.utils.make_grid(data,normalize=True)
  writer_fake.add_image("MNIST Fake Images",img_grid_fake,global_step=step)
  writer_real.add_image("MNIST Real Images",img_grid_real,global_step=step)
  step += 1